In [ ]:
# import libraries 
import pandas as pd
import numpy as np
import os

# Declare file path
factsale_file_path = "FactSale.csv"
dimcity_file_path = "DimCity.csv"
dimcustomer_file_path="DimCustomer.csv"
dimdate_file_path = "DimDate.csv"
dimemployee_file_path = "DimEmployee.xlsx"
dimstockitem_file_path = "DimStockItem.csv"
folder_name = "process_data"


def factSaleETL():
    """
    Function perform the basic preprocessing of factSales data and save into file 
    """
    try:
        print(f"Started processing file {factsale_file_path}")
        df_fact = pd.read_csv(factsale_file_path)

        # convert the Invoice Date key and Delivery Date Key into proper format
        # convert some values are like mm-dd-yyyy and mm/dd/yyyy make it to dd-mm-yyyy

        # Convert Invoice Date Key
        df_fact["Invoice Date Key"] = pd.to_datetime(
            df_fact["Invoice Date Key"].str.replace("-", "/"), 
            errors="coerce", 
            dayfirst=True
        ).dt.strftime("%Y-%m-%d")


        # Convert Delivery Date Key
        df_fact["Delivery Date Key"] = pd.to_datetime(
            df_fact["Delivery Date Key"].str.replace("-", "/"), 
            errors="coerce", 
            dayfirst=True
        ).dt.strftime("%Y-%m-%d")


        # rename columns
        df_fact.columns = df_fact.columns.str.replace(' ','')
        os.makedirs(folder_name, exist_ok=True)
        df_fact.to_csv(f"{folder_name}/FactSale.csv",index=False,encoding='utf-8',sep='|')
    
    except Exception as e:
        raise

def CityETL():
    """    
    Function perform the basic preprocessing of City data and save into file 

    """
    try:
        print(f"Started processing file {dimcity_file_path}")
        df_city = pd.read_csv(dimcity_file_path)

        # rename column
        df_city.columns = df_city.columns.str.replace(' ','')
        os.makedirs(folder_name,exist_ok=True)
        df_city.to_csv(f"{folder_name}/DimCity.csv",index=False,encoding='utf-8',sep='|')
    except Exception as e:
        raise

def CustomerETL():
    """    
    Function perform the basic preprocessing of Customer data and save into file 

    """
    try:
        print(f"Started processing file {dimcustomer_file_path}")
        df_cust = pd.read_csv(dimcustomer_file_path,skiprows=[0])

        df_cust.drop(index=0,axis=0,inplace=True)
        df_cust.drop(['Valid From','Valid To'],axis=1,inplace=True)
        # change datatype of postal code to int
        df_cust["Postal Code"] = pd.to_numeric(df_cust["Postal Code"], errors="coerce").astype("Int64")

        # clean "Credit Limit" column remove "?" AND "," while keeping other data as it is
        # convert datatype to float
        df_cust["Credit Limit"] = (
            df_cust["Credit Limit"]
            .str.replace("?", "", regex=False)   # remove the "?" symbol
            .str.replace(",", "", regex=False).str.replace("-","",regex=False)   # remove commas
            .astype(float)                       # convert to float
        )
        df_cust['Lineage Key'] = df_cust['Lineage Key'].astype("Int64")
        
        # rename columns
        df_cust.columns = df_cust.columns.str.replace(' ','')
        os.makedirs(folder_name, exist_ok=True)
        df_cust.to_csv(f"{folder_name}/DimCustomer.csv",index=False,encoding='utf-8',sep='|')
    except Exception as e:
        raise
    
def DateETL():
    """    
    Function perform the basic preprocessing of date data and save into file 

    """
    try:
        print(f"Started processing file {dimdate_file_path}")
        df_date = pd.read_csv(dimdate_file_path)
        
        # Convert Date into proper format
        df_date["Date"] = pd.to_datetime(df_date['Date'].str.replace('-','/'),format="%m/%d/%Y").dt.strftime('%Y-%m-%d')
        
        df_date.columns = df_date.columns.str.replace(' ','')
        # rename column
        df_date = df_date.rename(columns={'Date':'DateKey','Day':'DayOfMonth','Month':'MonthName'})
        
        os.makedirs(folder_name, exist_ok=True)
        df_date.to_csv(f"{folder_name}/DimDate.csv",index=False,encoding='utf-8',sep='|')
    
    except Exception as e:
        raise

def EmployeeETL():
    """    
    Function perform the basic preprocessing of Employee data and save into file 

    """
    try:
        print(f"Started processing file {dimemployee_file_path}")
        df_Emp = pd.read_excel(dimemployee_file_path)
        df_Emp.drop(['Valid From','Valid To'],axis=1,inplace=True)

        # rename columns
        df_Emp.columns = df_Emp.columns.str.replace(' ','')

        os.makedirs(folder_name, exist_ok=True)
        df_Emp.to_csv(f"{folder_name}/DimEmployee.csv",index=False,encoding='utf-8',sep='|')
    
    except Exception as e:
        raise
    
def StockItemETL():
    """    
    Function perform the basic preprocessing of StockItem data and save into file 

    """
    try:
        print(f"Started processing file {dimstockitem_file_path}")
        df_stock = pd.read_csv(dimstockitem_file_path)
        
        df_stock['Valid From'] = pd.to_datetime(df_stock['Valid From'],errors='coerce').dt.strftime('%Y-%m-%d')
        df_stock['Valid To'] = pd.to_datetime(df_stock['Valid To'],errors='coerce').dt.strftime('%Y-%m-%d')
        df_stock['Recommended Retail Price'] =df_stock['Recommended Retail Price'].str.replace('?','').str.replace('-','').str.replace(',','')
        
        # rename columns
        df_stock.columns = df_stock.columns.str.replace(' ','')

        os.makedirs(folder_name, exist_ok=True)
        df_stock.to_csv(f"{folder_name}/DimStockItem.csv",index=False,encoding='utf-8',sep='|')
          
    except Exception as e:
        raise
    
def main():
    try:
        vOutcome="data save Succesfully"
        factSaleETL()
        CityETL()
        CustomerETL()
        DateETL()
        EmployeeETL()
        StockItemETL()
    except Exception as e:
        vOutcome = f"Error in Main: Failed to save data with exception:{e}"
    else:
        vOutcome = f"ETL process Completed: Data save sccessfully."
    finally:
        
        print(vOutcome)
if __name__ == "__main__":
    main()


Started processing file FactSale.csv


C:\Users\ACER\AppData\Local\Temp\ipykernel_27784\3084572599.py:29: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_fact["Invoice Date Key"] = pd.to_datetime(
C:\Users\ACER\AppData\Local\Temp\ipykernel_27784\3084572599.py:37: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_fact["Delivery Date Key"] = pd.to_datetime(


Started processing file DimCity.csv
Started processing file DimCustomer.csv
Started processing file DimDate.csv
Started processing file DimEmployee.xlsx
Started processing file DimStockItem.csv
ETL process Completed: Data save sccessfully in file FactAndDimDataset.xlsx


d:\Downloads\DataModelling\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell H2 is marked as a date but the serial value 2958466 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Downloads\DataModelling\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell H196 is marked as a date but the serial value 2958466 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Downloads\DataModelling\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell H197 is marked as a date but the serial value 2958466 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Downloads\DataModelling\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell H198 is marked as a date but the serial value 2958466 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)
d:\Downloads\DataModelling\.venv\Lib\s